# Hugging Face Applications — Lesson 3: Question Answering

> Learning material for **Hugging Face Applications**. Companion to the lesson script `03_Question_Answering.py` (same content, runnable without Jupyter).

**Task ID:** HF-203  |  **Folder:** `documentation`


## What is (extractive) question answering?

You give the model a **context** (a passage) and a **question**, and it returns the answer **as a verbatim piece of the context**:

> 📄 Context: *"The Eiffel Tower was built in 1889 in Paris."*
>
> ❓ Question: *"In which city is the Eiffel Tower?"*
>
> 💬 Answer: *"Paris"* — a span found **inside** the passage.

This is called **extractive** QA, because the model *extracts* a span of text. (The other kind — *generative* QA — lets the model compose a free-form answer.)

## How the model finds the answer

An extractive QA model is a classifier over token *positions*:

1. The tokenizer builds one sequence: `[CLS] question [SEP] context [SEP]`
2. The model scores **every token** as a possible answer **start** → `start_logits`.
3. It scores every token as a possible answer **end** → `end_logits`.
4. Take the argmax of each; the tokens in between are the answer.

> **Analogy:** like a student underlining an answer in the exam text — first deciding where the underline *starts*, then where it *ends*.

## transformers v5 note

The `pipeline("question-answering")` shortcut was removed in v5. The model API below does exactly what it used to do internally.

**Step 1 — load the model** (DistilBERT fine-tuned on SQuAD, ~260 MB):


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "distilbert/distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
print("Loaded", model_name)


**Step 2 — ask!** Notice: question and context go **together** into one tokenizer call.


In [ ]:
question = "When was Hugging Face founded?"
context = (
    "Hugging Face was founded in 2016 in New York City. The company started as "
    "a chatbot app for teenagers, then pivoted to open-source machine learning "
    "tooling. Its Transformers library, released in 2018, quickly became one of "
    "the most popular libraries in the AI ecosystem."
)

inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

start = torch.argmax(outputs.start_logits, dim=-1).item()
end = torch.argmax(outputs.end_logits, dim=-1).item()
print(f"start token: {start}   end token: {end}")

answer_ids = inputs["input_ids"][0][start : end + 1]
answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
print(f"Answer: {answer}")


## Look inside: what is a *start logit*?

A logit is a raw score (not yet a probability). `argmax` picks the token with the highest start-score and the token with the highest end-score.

Let's see the scores for the first few tokens — higher = more likely:


In [ ]:
# The context tokens, with their start scores (first 12):
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
start_scores = torch.softmax(outputs.start_logits, dim=-1)[0]

for i, (tok, score) in enumerate(zip(tokens[:12], start_scores[:12])):
    print(f"{i:>3}  {tok:<20} start={score:.3f}")


## Edge cases

- **No answer in the text** — the scores will be flat/low; set a confidence threshold in real apps.
- **`end` before `start`** (rare) — clamp `end = start` (the script guards this).
- **Long context** — set `truncation=True`; the model has a token limit (~512 for DistilBERT).

## Try it yourself

1. Ask 3 more questions about the same context.
2. Replace the context with your own passage and question it.
3. Ask a question whose answer is NOT in the text — look at the scores.
4. Swap to `deepset/roberta-base-squad2` — stronger answers, slower.

## Summary

- Extractive QA = find a span. The model scores start and end positions.
- Question + context are one tokenizer input; answer = tokens in between.

**Next lesson:** HF-204 — Translation.  |  Extra reading: `../resources/reference_links.md`
